In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import KFold

In [ ]:
# Preparing the dataset
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

In [ ]:
df = pd.read_csv('course_lead_scoring_2026.csv')

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

numerical_columns = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
categorical_columns = ['lead_source', 'industry', 'employment_status', 'location']

numerical_columns = [c for c in df.select_dtypes(include="number").columns if c != "converted"]
categorical_columns = list(df.select_dtypes(exclude="number").columns)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')
columns_with_missing_values = df.isna().any()
columns_with_missing_values.sum()

In [ ]:
df[categorical_columns] = df[categorical_columns].fillna('NA')
df[numerical_columns] = df[numerical_columns].fillna(0.0)

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [ ]:
df.isnull().sum()

# Question 1

In [ ]:
y_train = df_train['converted'].values
for col in numerical_columns:
    auc = roc_auc_score(y_train, df_train[col])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[col])  # invert
    print(col, ":", auc)


# Question 2

In [ ]:
y_train = df_train['converted'].values
y_val   = df_val['converted'].values
del df_train['converted']
del df_val['converted']

In [ ]:
train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient="records")
val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient="records")
dv = DictVectorizer(sparse=False)
dv.fit(train_dict)

X_train = dv.transform(train_dict)
X_val   = dv.transform(val_dict)

model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]

auc = roc_auc_score(y_val, y_pred)

auc

# Question 3

In [ ]:
thresholds = np.linspace(0.0, 1.0, 101)

scores = []

for t in thresholds:
    converted_pred = (y_pred >= t).astype(int)

    tp = int(((converted_pred == 1) & (y_val == 1)).sum())
    fp = int(((converted_pred == 1) & (y_val == 0)).sum())
    fn = int(((converted_pred == 0) & (y_val == 1)).sum())
    tn = int(((converted_pred == 0) & (y_val == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    scores.append((t, tp, fp, fn, tn, precision, recall))

df_scores = pd.DataFrame(scores, columns=["threshold", "tp", "fp", "fn", "tn", "precision", "recall"])

df_scores.head()

plt.figure(figsize=(8, 5))
plt.plot(df_scores.threshold, df_scores.precision, label="Precision")
plt.plot(df_scores.threshold, df_scores.recall,    label="Recall")
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.title("Precision and Recall vs Threshold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
mask_non_zero = (df_scores.precision != 0) | (df_scores.recall != 0)
df_candidates = df_scores[mask_non_zero].copy()

df_candidates["abs_diff"] = (df_candidates.precision - df_candidates.recall).abs()

min_diff = df_candidates.abs_diff.min()
best_rows = df_candidates[df_candidates.abs_diff == min_diff]
best_threshold = best_rows.threshold.min()

print("Thresholds with the smalles |precision - recall|:")
print(best_rows[["threshold", "precision", "recall", "abs_diff"]].to_string(index=False))
print()
print(f">>> threshold = {best_threshold:.2f}")
df_candidates = df_scores[mask_non_zero].copy()

df_candidates["abs_diff"] = (df_candidates.precision - df_candidates.recall).abs()

min_diff = df_candidates.abs_diff.min()
best_rows = df_candidates[df_candidates.abs_diff == min_diff]
best_threshold = best_rows.threshold.min()

print("Thresholds with the smalles |precision - recall|:")
print(best_rows[["threshold", "precision", "recall", "abs_diff"]].to_string(index=False))
print()
print(f">>> threshold = {best_threshold:.2f}")

# Question 4

In [ ]:
thresholds = np.linspace(0.0, 1.0, 101)
scores = []
for t in thresholds:
    converted_pred = (y_pred >= t).astype(int)

    tp = int(((converted_pred == 1) & (y_val == 1)).sum())
    fp = int(((converted_pred == 1) & (y_val == 0)).sum())
    fn = int(((converted_pred == 0) & (y_val == 1)).sum())
    tn = int(((converted_pred == 0) & (y_val == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    scores.append((t, precision, recall, f1))

df_scores = pd.DataFrame(scores, columns=["threshold", "precision", "recall", "f1"])

max_f1 = df_scores.f1.max()
best_threshold = df_scores[df_scores.f1 == max_f1].threshold.min()

print(f"Threshold with maximal F1: {best_threshold:.2f}")

# Question 5

In [ ]:
# Split df în full_train + test (KFold goes only on full_train)
df = pd.read_csv('course_lead_scoring_2026.csv')

target = "converted"
numerical   = ["annual_income", "number_of_courses_viewed", "interaction_count", "lead_score"]
categorical = ["lead_source", "industry", "employment_status", "location"]

for c in categorical:
    df[c] = df[c].fillna("NA")
for c in numerical:
    df[c] = df[c].fillna(0.0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_full_train = df_full_train.reset_index(drop=True)

def train(df, y, C=1.0):
    cat = df[categorical + numerical].to_dict(orient="records")
    dv = DictVectorizer(sparse=False)
    dv.fit(cat)
    X = dv.transform(cat)
    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000)
    model.fit(X, y)
    return dv, model

def predict(df, dv, model):
    cat = df[categorical + numerical].to_dict(orient="records")
    X = dv.transform(cat)
    y_pred = model.predict_proba(X)[:, 1]
    return y_pred

kfold = KFold(n_splits=5, shuffle=True, random_state=1)

aucs = []
for train_idx, val_idx in kfold.split(df_full_train):
    df_train_fold = df_full_train.iloc[train_idx]
    df_val_fold   = df_full_train.iloc[val_idx]

    y_train_fold = df_train_fold[target].values
    y_val_fold   = df_val_fold[target].values

    df_train_fold = df_train_fold.drop(columns=[target])
    df_val_fold   = df_val_fold.drop(columns=[target])

    dv, model = train(df_train_fold, y_train_fold, C=1.0)
    y_pred_fold = predict(df_val_fold, dv, model)

    auc = roc_auc_score(y_val_fold, y_pred_fold)
    aucs.append(auc)

print(f"Mean AUC: {np.mean(aucs):.6f}")
print(f"Std  AUC: {np.std(aucs):.6f}")
print(f"Std rounded to 3 digits: {round(np.std(aucs), 3)}")

# Question 6

In [ ]:
def train(df, y, C):
    cat = df[categorical + numerical].to_dict(orient="records")
    dv = DictVectorizer(sparse=False)
    dv.fit(cat)
    X = dv.transform(cat)
    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000)
    model.fit(X, y)
    return dv, model

def predict(df, dv, model):
    cat = df[categorical + numerical].to_dict(orient="records")
    X = dv.transform(cat)
    y_pred = model.predict_proba(X)[:, 1]
    return y_pred

kfold = KFold(n_splits=5, shuffle=True, random_state=1)

C_values = [0.000001, 0.001, 1]
results = []

print(f"{'C':>10} | {'mean AUC':>9} | {'std AUC':>9} | per-fold AUCs")
print("-" * 70)

for C in C_values:
    aucs = []
    for train_idx, val_idx in kfold.split(df_full_train):
        df_train_fold = df_full_train.iloc[train_idx]
        df_val_fold   = df_full_train.iloc[val_idx]

        y_train_fold = df_train_fold[target].values
        y_val_fold   = df_val_fold[target].values

        # Remove target from feature frames
        df_train_fold = df_train_fold.drop(columns=[target])
        df_val_fold   = df_val_fold.drop(columns=[target])

        # Train + predict
        dv, model = train(df_train_fold, y_train_fold, C)
        y_pred_fold = predict(df_val_fold, dv, model)

        # Evaluate AUC
        auc = roc_auc_score(y_val_fold, y_pred_fold)
        aucs.append(auc)

    mean_auc = np.mean(aucs)
    std_auc  = np.std(aucs)
    results.append({
        "C": C,
        "mean": round(mean_auc, 3),
        "std":  round(std_auc, 3),
        "mean_raw": mean_auc,
        "std_raw":  std_auc,
        "aucs": aucs,
    })

    print(f"{C:>10.6f} | {mean_auc:>9.4f} | {std_auc:>9.4f} | "
          + ", ".join(f"{a:.4f}" for a in aucs))

print()
print("=" * 70)
print("Summary (mean & std rounded to 3 decimal digits):")
df_res = pd.DataFrame([
    {"C": r["C"], "mean": r["mean"], "std": r["std"]}
    for r in results
])
print(df_res.to_string(index=False))
print()

df_res_sorted = df_res.sort_values(
    by=["mean", "std", "C"],
    ascending=[False, True, True]
)
print("Sorted by (mean desc, std asc, C asc):")
print(df_res_sorted.to_string(index=False))
print()

best = df_res_sorted.iloc[0]
print(f">>> Best C = {best['C']}")
print(f"    mean AUC (rounded) = {best['mean']}")
print(f"    std  AUC (rounded) = {best['std']}")